# 146 — ORPO Alignment

**What you'll learn:**
- The alignment landscape: RLHF → DPO → ORPO and what problem each solves
- Why ORPO needs no reference model — and what that saves in memory and complexity
- The ORPO loss function: L_SFT + lambda × L_OR
- How the beta (lambda) hyperparameter controls the preference strength
- The preference dataset format: {prompt, chosen, rejected} triples
- How to measure win rate (preference rate) before and after training

---
**Source:** `examples/146-orpo-alignment/`  
**Model:** `HuggingFaceTB/SmolLM2-135M-Instruct`  
**Part A** is CPU-safe. **Part B** requires a CUDA GPU with 4GB+ VRAM.

In [ ]:
%pip install -q trl transformers peft datasets torch

## Part A — CPU-Safe Concept Demonstrations

All cells below run without a GPU and without downloading any model weights.

### A1 — The Alignment Landscape: RLHF → DPO → ORPO

All alignment methods teach a model to prefer 'good' responses over 'bad' ones. They differ in how they define 'good' and how they enforce that preference.

| Method | Reference model? | Reward model? | Training phases | Memory overhead | Notes |
|--------|-----------------|---------------|-----------------|-----------------|-------|
| RLHF | Yes (SFT as KL anchor) | Yes (separate model) | 3 (SFT → RM → PPO) | 2-3× base model | The classic approach; complex |
| DPO | Yes (frozen SFT copy) | No | 1 (preference learning) | 2× base model | Simpler than RLHF; ref model stays in memory |
| ORPO | No | No | 1 (SFT + preference together) | 1× base model | Newest; smallest memory footprint |

```
RLHF (2022): Instruct the model → train a reward model → use PPO to optimize against it
Problem: Three separate training phases, requires PPO infrastructure, reward model can overfit

DPO (2023): Skip the reward model — optimize directly on (chosen, rejected) pairs
Problem: Still needs a frozen reference model in memory for KL regularization (~2× VRAM)

ORPO (2024): Skip the reference model too — use relative log-odds within the current model
Advantage: One model, one training phase, ~1× VRAM, simpler implementation
```

### A2 — The ORPO Loss Function

```
ORPO loss:
  L_ORPO = L_SFT + beta × L_OR

where:
  L_SFT = cross-entropy loss on chosen response (standard language model loss)
  L_OR  = -log(sigmoid(log_odds_ratio_chosen - log_odds_ratio_rejected))
  beta  = hyperparameter controlling the strength of the preference signal

Log-odds ratio:
  log_odds(response) = log(P(response) / (1 - P(response)))
                     = logit(P(response))
                     ≈ -loss (since loss = -log P and P is a sequence probability)

So the odds ratio term penalizes the model when:
  log_odds_chosen < log_odds_rejected
  i.e., when the model assigns higher probability to the rejected response

Why no reference model?
DPO computes: log P_policy(chosen) / P_ref(chosen) - log P_policy(rejected) / P_ref(rejected)
ORPO computes: log_odds P_policy(chosen) - log_odds P_policy(rejected)

ORPO replaces the reference model's role with the odds ratio (which self-normalizes within the current model).
```

**Practical note on odds vs probability:**  
Odds ratio is log(p/(1-p)). When p is small (as token probabilities are), log(p/(1-p)) ≈ log(p) ≈ -loss. So in practice, ORPO's preference term is approximately: sigma(loss_rejected - loss_chosen) — minimized when chosen loss is lower.

In [ ]:
# A3 — Odds ratio formula walkthrough (pure Python, no model)
# Given two log-probability sequences, compute the log-odds ratio manually.

import math

def log_prob_to_odds(log_prob: float) -> float:
    """Convert log probability to log-odds: log(p / (1 - p))"""
    p = math.exp(log_prob)
    p = max(1e-9, min(1 - 1e-9, p))  # clamp for numerical stability
    return math.log(p / (1 - p))

def orpo_or_loss(log_prob_chosen: float, log_prob_rejected: float) -> float:
    """
    ORPO odds ratio loss (scalar version for demonstration).
    Real implementation averages over the full sequence.
    """
    lo_chosen   = log_prob_to_odds(log_prob_chosen)
    lo_rejected = log_prob_to_odds(log_prob_rejected)
    # sigmoid(log_odds_chosen - log_odds_rejected) = P(model prefers chosen)
    odds_diff = lo_chosen - lo_rejected
    p_preferred = 1 / (1 + math.exp(-odds_diff))
    # Loss = -log(P(prefers chosen)) — minimized when chosen is strongly preferred
    return -math.log(p_preferred + 1e-9)

# Demonstrate with three scenarios:
scenarios = [
    ("Untrained (random, equal)",    -2.5,  -2.5),
    ("Slightly prefers chosen",      -2.3,  -2.7),
    ("Strongly prefers chosen",      -1.5,  -3.5),
    ("Prefers rejected (bad)",       -3.5,  -1.5),
]

print(f"{'Scenario':<35} {'log_p_chosen':<15} {'log_p_rejected':<17} {'OR loss':<10} {'Preferred?'}")
print("-" * 85)

for name, lpc, lpr in scenarios:
    loss = orpo_or_loss(lpc, lpr)
    preferred = "chosen" if lpc > lpr else "rejected" if lpr > lpc else "neither"
    print(f"{name:<35} {lpc:<15.1f} {lpr:<17.1f} {loss:<10.4f} {preferred}")

print()
print("Interpretation:")
print("  OR loss is minimized when chosen log-prob >> rejected log-prob.")
print("  Gradient signal: increase log_prob(chosen) AND decrease log_prob(rejected).")
print("  This is the key difference from SFT: SFT only increases P(chosen).")

In [ ]:
# A4 — Preference dataset format: {prompt, chosen, rejected}
# Shows what makes a good vs bad (rejected) response.

PREFERENCE_DATA = [
    {
        "prompt": "How do you approach solving a complex problem?",
        "chosen": (
            "I break it into smaller subproblems, identify the core constraint, "
            "and solve each part systematically before integrating the solution."
        ),
        "rejected": "I dunno, maybe try something?",
    },
    {
        "prompt": "What is 6 times 7?",
        "chosen": "The answer is 42. This is computed by multiplying 6 × 7 = 42.",
        "rejected": "The answer could be many things.",
    },
    {
        "prompt": "How do you reverse a list in Python?",
        "chosen": (
            "Use `my_list[::-1]` for a new reversed list, "
            "or `my_list.reverse()` to reverse in place."
        ),
        "rejected": "You can do it lots of ways I guess.",
    },
    {
        "prompt": "What is the capital of France?",
        "chosen": "The capital of France is Paris, which has been the country's capital since 987 AD.",
        "rejected": "France has a capital city somewhere in Europe.",
    },
    {
        "prompt": "How do you commit changes in git?",
        "chosen": "Use `git commit -m 'message'` to commit staged changes with a descriptive message.",
        "rejected": "Just run some git command or something.",
    },
]

print(f"Preference dataset: {len(PREFERENCE_DATA)} rows\n")
print(f"Schema: {{'prompt': str, 'chosen': str, 'rejected': str}}\n")

for i, row in enumerate(PREFERENCE_DATA[:3]):
    print(f"--- Row {i} ---")
    print(f"  prompt  : {row['prompt']}")
    print(f"  chosen  : {row['chosen'][:80]}")
    print(f"  rejected: {row['rejected']}")
    print()

print("What makes a good 'chosen' response:")
print("  - Specific, factual, actionable")
print("  - Correct, not vague or wrong")
print("  - Helpful to the human asking")
print()
print("What makes a good 'rejected' response:")
print("  - Should be plausibly wrong or unhelpful (not obviously random)")
print("  - Real-world rejected responses come from models or annotators")
print("  - Too-easy rejections ('I don't know') give weak training signal")

### A5 — The beta Hyperparameter

```
L_ORPO = L_SFT + beta × L_OR

beta controls how strongly ORPO pushes the model to prefer chosen over rejected.
```

| beta value | Effect | When to use |
|------------|--------|-------------|
| 0.0 | Pure SFT — ignores rejected responses entirely | Baseline; no alignment signal |
| 0.05 | Very light alignment | When SFT quality must dominate |
| 0.1 | Default (TRL's ORPOConfig default) | Good starting point for most tasks |
| 0.25 | Moderate alignment | When model still generates bad responses |
| 0.5 | Strong alignment | Heavy preference enforcement |
| 1.0+ | Very aggressive | May hurt fluency and coherence |

**Analogy:** Beta is like a dial between 'learn to generate chosen responses' (beta=0) and 'aggressively avoid rejected responses' (beta=∞). The TRL default of 0.1 was tuned empirically on the ORPO paper's benchmark — it keeps fluency high while adding meaningful alignment.

**Connection to DPO:** In DPO, the equivalent hyperparameter is the KL penalty coefficient. ORPO's beta serves the same mathematical role — controlling regularization strength — but doesn't need a reference model to compute the KL term.

In [ ]:
# A6 — Simulate how beta affects SFT loss vs OR loss balance
# Shows the tradeoff without loading any model.

def simulate_orpo_loss(sft_loss: float, or_loss: float, beta: float) -> dict:
    """Compute total ORPO loss and show the contribution breakdown."""
    total = sft_loss + beta * or_loss
    sft_pct = 100 * sft_loss / total if total > 0 else 0
    or_pct  = 100 * (beta * or_loss) / total if total > 0 else 0
    return {
        "beta": beta,
        "sft_loss": sft_loss,
        "or_term": beta * or_loss,
        "total": total,
        "sft_pct": round(sft_pct, 1),
        "or_pct": round(or_pct, 1),
    }

# Assume fixed base losses from a partially trained model
BASE_SFT_LOSS = 1.8
BASE_OR_LOSS  = 0.9  # OR loss at the same checkpoint

betas = [0.0, 0.05, 0.1, 0.25, 0.5, 1.0]

print("Beta sweep: L_ORPO = L_SFT + beta × L_OR")
print(f"(Base L_SFT={BASE_SFT_LOSS}, Base L_OR={BASE_OR_LOSS})\n")
print(f"{'beta':<8} {'L_SFT':<10} {'beta×L_OR':<12} {'Total':<10} {'SFT %':<10} {'OR %'}")
print("-" * 58)

for beta in betas:
    r = simulate_orpo_loss(BASE_SFT_LOSS, BASE_OR_LOSS, beta)
    print(f"{r['beta']:<8.2f} {r['sft_loss']:<10.3f} {r['or_term']:<12.3f} {r['total']:<10.3f} {r['sft_pct']:<10.1f} {r['or_pct']}")

print()
print("At beta=0.1 (default): OR term contributes only ~4.8% of total loss.")
print("This preserves language model fluency while adding alignment signal.")
print("At beta=0.5: OR term grows to ~20% — much stronger preference enforcement.")

In [ ]:
# A7 — Complexity and memory comparison across alignment methods
# Shows memory multipliers and training stage counts (no model loaded).

METHODS = {
    "RLHF": {
        "stages": ["Stage 1: SFT on demonstrations", "Stage 2: Reward model training",
                   "Stage 3: PPO fine-tuning"],
        "models_in_memory": "SFT policy + reference + reward model (3 models)",
        "memory_multiplier": 3.0,
        "reference_model": True,
        "reward_model": True,
        "complexity": "High — PPO is an RL algorithm with clipping, GAE, value function",
    },
    "DPO": {
        "stages": ["Stage 1: SFT fine-tuning", "Stage 2: DPO preference optimization"],
        "models_in_memory": "Policy + frozen reference (2 models)",
        "memory_multiplier": 2.0,
        "reference_model": True,
        "reward_model": False,
        "complexity": "Medium — direct gradient update, no PPO",
    },
    "ORPO": {
        "stages": ["Stage 1: SFT + preference optimization (combined)"],
        "models_in_memory": "Policy only (1 model)",
        "memory_multiplier": 1.0,
        "reference_model": False,
        "reward_model": False,
        "complexity": "Low — one-stage training, no RL infrastructure",
    },
}

BASE_MODEL_GB = 3.5  # e.g., SmolLM2-135M with bf16 overhead

print("Alignment method comparison\n")
for method, info in METHODS.items():
    print(f"{'='*50}")
    print(f"Method: {method}")
    print(f"  Stages       : {len(info['stages'])}")
    for s in info['stages']:
        print(f"    - {s}")
    print(f"  In memory    : {info['models_in_memory']}")
    print(f"  Memory × base: {info['memory_multiplier']}× (~{BASE_MODEL_GB * info['memory_multiplier']:.1f} GB for 135M model)")
    print(f"  Ref model    : {'Yes' if info['reference_model'] else 'No'}")
    print(f"  Reward model : {'Yes' if info['reward_model'] else 'No'}")
    print(f"  Complexity   : {info['complexity']}")
    print()

print("Key takeaway:")
print("  ORPO saves ~2-3× memory vs RLHF and eliminates reward model training.")
print("  Quality: RLHF ≥ DPO ≈ ORPO (within ~1-2% on most benchmarks).")
print("  For small models and limited compute, ORPO is the pragmatic choice.")

---
## Exercise 1: Response Quality Classifier

**Problem:** Write a function that classifies a candidate response as `"chosen"` or `"rejected"` based on two heuristics:

1. **Length heuristic**: responses shorter than 10 words are "rejected" (too brief to be useful)
2. **Refusal heuristic**: responses containing any of these phrases are "rejected":
   - `"I don't know"`
   - `"I'm not sure"`
   - `"I cannot"`
   - `"I'm unable"`
   - `"I dunno"`

Otherwise, classify as `"chosen"`.

This simulates a simple auto-labeling function you'd use to generate preference data from model outputs.

In [ ]:
# Exercise 1: Classify response as 'chosen' or 'rejected'

REFUSAL_PHRASES = [
    "I don't know",
    "I'm not sure",
    "I cannot",
    "I'm unable",
    "I dunno",
]

def classify_response(response: str) -> str:
    """
    Returns 'chosen' or 'rejected' based on:
    - Length < 10 words → 'rejected'
    - Contains a refusal phrase → 'rejected'
    - Otherwise → 'chosen'
    """
    # TODO: implement this function
    pass


# Test cases
test_responses = [
    "I dunno, maybe try something?",
    "Gradient descent is an optimization algorithm that iteratively minimizes a loss function.",
    "I cannot help with that request.",
    "Use git commit -m to save your changes.",
    "Yes.",  # too short
    "The capital of France is Paris, which has been France's capital since 987 AD.",
    "I'm not sure about the specifics of this topic.",
]

for resp in test_responses:
    label = classify_response(resp)
    print(f"[{label.upper():<8}] {resp[:70]}")

In [ ]:
# Exercise 1 — Answer Key

REFUSAL_PHRASES = [
    "I don't know",
    "I'm not sure",
    "I cannot",
    "I'm unable",
    "I dunno",
]

def classify_response(response: str) -> str:
    """
    Classify response as 'chosen' or 'rejected'.

    Rules applied in order:
    1. Refusal phrases → rejected (check before length, some refusals are long)
    2. Too short (<10 words) → rejected
    3. Otherwise → chosen
    """
    # Check for refusal phrases first (case-insensitive)
    response_lower = response.lower()
    for phrase in REFUSAL_PHRASES:
        if phrase.lower() in response_lower:
            return "rejected"

    # Length heuristic
    word_count = len(response.split())
    if word_count < 10:
        return "rejected"

    return "chosen"

test_responses = [
    "I dunno, maybe try something?",
    "Gradient descent is an optimization algorithm that iteratively minimizes a loss function.",
    "I cannot help with that request.",
    "Use git commit -m to save your changes.",
    "Yes.",
    "The capital of France is Paris, which has been France's capital since 987 AD.",
    "I'm not sure about the specifics of this topic.",
]

for resp in test_responses:
    label = classify_response(resp)
    print(f"[{label.upper():<8}] {resp[:70]}")

print()
# This classifier can be used to auto-generate preference datasets:
# 1. Run a weak model on prompts to get candidate responses
# 2. Classify each as chosen/rejected using rules like these
# 3. Use higher-quality responses as chosen, lower-quality as rejected
# Production classifiers also use: factual accuracy checks, reward model scores,
# human preference labels (Likert scale), or LLM-as-judge scoring.
print("Usage: run a model on prompts, classify outputs, build preference pairs.")
print("More sophisticated classifiers use LLM-as-judge or reward model scoring.")

---
## Exercise 2: Convert Reward-Scored Data to ORPO Format

**Problem:** You have a dataset in RLHF format:
```python
[
    {"prompt": "...", "responses": [{"text": "...", "reward": 2.3}, {"text": "...", "reward": 1.1}]},
    ...
]
```

Write a function `rlhf_to_orpo(rlhf_rows)` that converts this to ORPO format:
```python
[{"prompt": "...", "chosen": "...", "rejected": "..."}]
```

Rules:
- The response with the **highest reward score** becomes `chosen`
- The response with the **lowest reward score** becomes `rejected`
- If a row has fewer than 2 responses, skip it

In [ ]:
# Exercise 2: Convert RLHF reward-scored data to ORPO preference format

RLHF_DATA = [
    {
        "prompt": "How do you handle errors in Python?",
        "responses": [
            {"text": "Use try/except blocks to catch and handle specific exceptions.", "reward": 3.2},
            {"text": "Errors are bad, avoid them.", "reward": 0.8},
            {"text": "Python has built-in exception handling with try/except/finally blocks.", "reward": 3.8},
        ],
    },
    {
        "prompt": "What is a REST API?",
        "responses": [
            {"text": "A REST API is a web service following REST architectural constraints.", "reward": 2.9},
            {"text": "It's some kind of internet thing.", "reward": 0.5},
        ],
    },
    {
        "prompt": "Single response",
        "responses": [
            {"text": "Only one response here.", "reward": 1.5},
        ],
    },
]

def rlhf_to_orpo(rlhf_rows: list) -> list:
    """
    Convert RLHF reward-scored data to ORPO preference format.
    chosen = highest reward response
    rejected = lowest reward response
    Skip rows with fewer than 2 responses.
    """
    # TODO: implement this function
    pass

orpo_dataset = rlhf_to_orpo(RLHF_DATA)

print(f"Input rows: {len(RLHF_DATA)}, Output rows: {len(orpo_dataset)}\n")
for row in orpo_dataset:
    print(f"Prompt  : {row['prompt']}")
    print(f"Chosen  : {row['chosen'][:80]}")
    print(f"Rejected: {row['rejected'][:80]}")
    print()

In [ ]:
# Exercise 2 — Answer Key

def rlhf_to_orpo(rlhf_rows: list) -> list:
    """Convert RLHF reward-scored data to ORPO preference format."""
    orpo_rows = []
    for row in rlhf_rows:
        responses = row.get("responses", [])

        # Skip rows with fewer than 2 responses
        if len(responses) < 2:
            continue

        # Sort by reward score
        sorted_responses = sorted(responses, key=lambda r: r["reward"])

        orpo_rows.append({
            "prompt":   row["prompt"],
            "chosen":   sorted_responses[-1]["text"],   # highest reward
            "rejected": sorted_responses[0]["text"],    # lowest reward
        })

    return orpo_rows

orpo_dataset = rlhf_to_orpo(RLHF_DATA)

print(f"Input rows: {len(RLHF_DATA)} → Output rows: {len(orpo_dataset)} (1 skipped: <2 responses)\n")
for i, row in enumerate(orpo_dataset):
    print(f"--- Row {i} ---")
    print(f"  Prompt  : {row['prompt']}")
    print(f"  Chosen  : {row['chosen'][:80]}")
    print(f"  Rejected: {row['rejected'][:80]}")
    print()

# Why the reward gap matters:
# A large reward gap (3.8 vs 0.8) gives a strong learning signal.
# A small gap (2.9 vs 2.7) gives a weak signal and may confuse training.
# Best practice: filter out pairs where |reward_chosen - reward_rejected| < threshold.
print("Best practice: filter pairs where |reward_chosen - reward_rejected| < 1.0")
print("Weak pairs dilute the training signal and slow convergence.")

---
## Part B — Full ORPO Training Run

> **Requirements:**
> - CUDA GPU with at least **4 GB VRAM** (T4, RTX 3060, or better)
> - Free option: [Google Colab T4 runtime](https://colab.research.google.com) → Runtime > Change runtime type > T4 GPU
> - Cells will raise `EnvironmentError` on CPU-only machines — that is expected

**What Part B demonstrates:**
- Loading a model and measuring preference rate (win rate) before training
- Running ORPO with TRL's `ORPOTrainer`
- Printing SFT loss and OR loss separately at each logging step
- Measuring win rate improvement after training

In [ ]:
# B1 — GPU fail-fast check
import torch

if not torch.cuda.is_available():
    raise EnvironmentError(
        "No CUDA GPU detected.\n"
        "Part B requires a GPU with 4GB+ VRAM.\n"
        "On Colab: Runtime > Change runtime type > T4 GPU (free tier).\n"
        "Re-run this cell after switching runtimes."
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU detected : {gpu_name}")
print(f"Total VRAM   : {vram_gb:.1f} GB")

if vram_gb < 4:
    raise EnvironmentError(
        f"GPU has only {vram_gb:.1f} GB VRAM. ORPO training requires at least 4 GB."
    )
print("GPU check passed. Ready for Part B.")

In [ ]:
# B2 — Full ORPO training pipeline (from workflow.py)
import sys
sys.path.insert(0, '/content')  # Colab; adjust if running locally

from src.workflow import create_workflow

workflow = create_workflow()

state = {
    "model_name": "HuggingFaceTB/SmolLM2-135M-Instruct",
    "n_train_pairs": 20,
    "max_steps": 30,
    "preference_rate_before": 0.0,
    "preference_rate_after": 0.0,
    "final_loss": 0.0,
}

print(f"Model    : {state['model_name']}")
print(f"Pairs    : {state['n_train_pairs']}")
print(f"Steps    : {state['max_steps']}")
print(f"Method   : ORPO (beta=0.1, one-stage SFT + preference)")
print()

result = workflow(state)

before = result['preference_rate_before']
after  = result['preference_rate_after']
delta  = after - before

print()
print("=" * 50)
print("ORPO Training Results")
print("=" * 50)
print(f"Preference rate before : {before:.1%}")
print(f"Preference rate after  : {after:.1%}")
print(f"Improvement            : {delta:+.1%}")
print(f"Final loss             : {result['final_loss']}")
print()
print("Preference rate = fraction of (chosen, rejected) pairs where")
print("the model assigns lower loss to chosen than to rejected.")
print(f"A random model scores ~50%. After ORPO, expect 70-90%+ on the training data.")

In [ ]:
# B3 — Show SFT loss and OR loss contributions separately
# Uses a fresh model load to demonstrate the loss breakdown.

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from src.tools import build_preference_dataset, eval_preference_rate, get_orpo_config
from trl import ORPOTrainer

MODEL_NAME = "HuggingFaceTB/SmolLM2-135M-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
dataset = build_preference_dataset(10)

# Evaluate before
rate_before = eval_preference_rate(model, tokenizer, dataset, n=5)
print(f"Preference rate BEFORE: {rate_before:.1%}\n")

# Train with verbose logging to show loss components
orpo_cfg = get_orpo_config("/tmp/orpo_b3", max_steps=20)
orpo_cfg.logging_steps = 5   # print every 5 steps

trainer = ORPOTrainer(
    model=model,
    args=orpo_cfg,
    train_dataset=dataset,
    tokenizer=tokenizer,
)

print("Training with ORPO (20 steps)...")
print("Watch for 'sft_loss' and 'odds_ratio_loss' in the log:\n")
train_result = trainer.train()

rate_after = eval_preference_rate(model, tokenizer, dataset, n=5)
print(f"\nPreference rate AFTER  : {rate_after:.1%}")
print(f"Improvement            : {rate_after - rate_before:+.1%}")
print(f"Final training loss    : {round(train_result.training_loss, 4)}")
print()
print("TRL's ORPOTrainer logs 'sft_loss' and 'odds_ratio_loss' separately.")
print("This lets you verify the preference signal is active (OR loss > 0).")

### Preference Annotation Guidelines

The quality of an alignment model is bounded by the quality of its preference annotations.

**Good annotation guidelines specify:**

| Dimension | What to evaluate | When to mark as chosen |
|---|---|---|
| Accuracy | Is the information factually correct? | All facts verifiable or marked as uncertain |
| Completeness | Does it answer the full question? | Addresses all parts of the prompt |
| Safety | Are there harmful elements? | No harmful content, appropriate caveats |
| Format | Is the structure appropriate? | Matches expected length and format |
| Tone | Does it match the use case? | Professional, empathetic, or casual as appropriate |

**Inter-annotator agreement (IAA)** measures consistency: aim for κ > 0.7 (Cohen's kappa).
Low IAA = noisy labels = weak alignment signal.

In [ ]:
# SimPO vs ORPO vs DPO: key differences in loss formulation
# SimPO (Simple Preference Optimization) is another reference-free method (2024)
# All three avoid RLHF's complexity; DPO still needs a reference model

METHODS = {
    'RLHF': {
        'reference_model': True,
        'reward_model': True,
        'training_phases': 3,
        'memory_factor': '3x base',
        'complexity': 'Very high',
    },
    'DPO': {
        'reference_model': True,
        'reward_model': False,
        'training_phases': 2,
        'memory_factor': '2x base',
        'complexity': 'Moderate',
    },
    'ORPO': {
        'reference_model': False,
        'reward_model': False,
        'training_phases': 1,
        'memory_factor': '1x base',
        'complexity': 'Low',
    },
    'SimPO': {
        'reference_model': False,
        'reward_model': False,
        'training_phases': 1,
        'memory_factor': '1x base',
        'complexity': 'Low',
    },
}

print(f'{'Method':>8} | {'ref model':>9} | {'phases':>6} | {'memory':>10} | {'complexity':>12}')
print('-' * 58)
for name, m in METHODS.items():
    ref = 'Yes' if m['reference_model'] else 'No'
    print(f'{name:>8} | {ref:>9} | {m["training_phases"]:>6} | {m["memory_factor"]:>10} | {m["complexity"]:>12}')

print()
print('ORPO and SimPO are the two most practical choices for single-GPU alignment.')
print('SimPO uses average log-probability over sequence length (length-normalized).')

### Chosen/Rejected Ratio and Dataset Size

Unlike SFT which can overfit slowly, preference data quality matters more than quantity:

| Dataset size | Expected outcome | Notes |
|---|---|---|
| < 500 pairs | Weak alignment | Only use for domain-specific niche tasks |
| 500–2,000 | Moderate alignment | Viable for targeted behavior changes |
| 2,000–10,000 | Strong alignment | Sweet spot for most use cases |
| > 10,000 | Diminishing returns | More impactful to improve label quality |

**Chosen:Rejected ratio = 1:1** is standard — one pair per prompt.
Some datasets include multiple rejected responses per prompt (harder negatives), which can help
but requires averaging the rejection loss across candidates.

In [ ]:
# Analyze preference dataset statistics to spot quality issues
from statistics import mean, stdev

def analyze_preference_dataset(data: list) -> dict:
    """
    data: list of {prompt, chosen, rejected}
    Returns stats useful for spotting annotation issues.
    """
    chosen_lens = [len(d['chosen'].split()) for d in data]
    rejected_lens = [len(d['rejected'].split()) for d in data]
    prompt_lens = [len(d['prompt'].split()) for d in data]
    # Length ratio > 3 or < 0.33 may indicate trivial rejections
    length_ratios = [c / (r + 1) for c, r in zip(chosen_lens, rejected_lens)]
    trivial_count = sum(1 for r in length_ratios if r > 3 or r < 0.33)
    identical = sum(1 for d in data if d['chosen'].strip() == d['rejected'].strip())
    return {
        'total_pairs': len(data),
        'avg_chosen_len': round(mean(chosen_lens), 1),
        'avg_rejected_len': round(mean(rejected_lens), 1),
        'avg_prompt_len': round(mean(prompt_lens), 1),
        'trivial_pairs': trivial_count,
        'identical_pairs': identical,
    }

sample_data = [
    {'prompt': 'What is Python?', 'chosen': 'Python is a high-level interpreted language created by Guido van Rossum, known for readability.', 'rejected': 'I cannot answer that.'},
    {'prompt': 'Explain recursion.', 'chosen': 'Recursion is when a function calls itself with a smaller input until a base case is reached.', 'rejected': 'Recursion is when a function calls itself.'},
    {'prompt': 'What is 2+2?', 'chosen': '4', 'rejected': 'I don\'t know.'},
    {'prompt': 'What is the capital of France?', 'chosen': 'Paris.', 'rejected': 'Paris.'},  # identical pair!
]
stats = analyze_preference_dataset(sample_data)
for k, v in stats.items(): print(f'{k:>20}: {v}')
print()
if stats['identical_pairs'] > 0:
    print(f'WARNING: {stats["identical_pairs"]} identical chosen/rejected pairs — remove these!')
if stats['trivial_pairs'] > 0:
    print(f'INFO: {stats["trivial_pairs"]} pairs with extreme length ratios — review for trivial rejections.')

---
## What's Next

- **ORPO paper** (Hong et al. 2024, arxiv.org/abs/2403.07691) — original paper with ablations
- **SimPO** (Meng et al. 2024, arxiv.org/abs/2405.14734) — another reference-free method
- **TRL ORPOTrainer** (huggingface.co/docs/trl/orpo_trainer) — 10-line ORPO training
- **Anthropic Constitutional AI paper** — the theoretical basis for preference-based alignment
- **Example 145 — LoRA Architecture Ablation:** choose the adapter config before running alignment

---
## What's Next

You've trained a model with ORPO — eliminating the reference model and combining SFT with preference learning in a single pass. Here's where to go deeper:

- **ORPO paper**: Hong et al. (2024) — "ORPO: Monolithic Preference Optimization without Reference Model" — https://arxiv.org/abs/2403.07691
- **TRL ORPOTrainer docs**: https://huggingface.co/docs/trl/orpo_trainer — includes `ORPOConfig` parameters and dataset format requirements
- **DPO paper** (for comparison): Rafailov et al. (2023) — "Direct Preference Optimization" — https://arxiv.org/abs/2305.18290
- **Preference data datasets**: `Anthropic/hh-rlhf`, `HuggingFaceH4/ultrafeedback_binarized`, `Intel/orca_dpo_pairs` on HuggingFace Hub
- **Example 144**: QLoRA Fine-Tuning — combine QLoRA's memory efficiency with ORPO to align a quantized model
- **Example 147**: Model Merging — take two ORPO-aligned variants and SLERP-merge them to get a single model that combines both specializations

---
## Summary

| Concept | Key detail |
|---------|-----------|
| ORPO vs DPO vs RLHF | ORPO: no reference model, 1× memory, 1 training phase |
| ORPO loss | L_ORPO = L_SFT + beta × L_OR |
| OR loss | -log(sigmoid(log_odds_chosen - log_odds_rejected)) |
| beta default | 0.1 (TRL default) — start here, increase if alignment is weak |
| Dataset format | {prompt, chosen, rejected} — same as DPO |
| Win rate | P(loss_chosen < loss_rejected) — baseline ~50%, target 70%+ |